# Stage 4 Modeling — Transplant Receipt
## Logistic Regression: Pre and Post eGFR Era

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report
import joblib

# load model-ready datasets
df_pre = pd.read_csv('../data/pre_egfr_model.csv')
df_post = pd.read_csv('../data/post_egfr_model.csv')

print(f"Pre-eGFR: {df_pre.shape}")
print(f"Post-eGFR: {df_post.shape}")

print(f"\nPre-eGFR transplant rate: {df_pre['RECEIVED_TX'].mean():.1%}")
print(f"Post-eGFR transplant rate: {df_post['RECEIVED_TX'].mean():.1%}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

covariates = [
    'CAN_RACE_SRTR',
    'INSURANCE_GRP',
    'CAN_GENDER',
    'CAN_AGE_AT_LISTING',
    'CAN_ABO',
    'PREEMPTIVE',
    'CAN_DGN',
]

def prepare_features(df):
    X = pd.get_dummies(df[covariates],
                       columns=['CAN_RACE_SRTR', 'INSURANCE_GRP',
                                'CAN_GENDER', 'CAN_ABO'],
                       drop_first=True)
    y = df['RECEIVED_TX']
    return X, y

X_pre, y_pre = prepare_features(df_pre)
X_post, y_post = prepare_features(df_post)

# fit models
# scaling pipeline
model_pre = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

model_post = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

model_pre.fit(X_pre, y_pre)
model_post.fit(X_post, y_post)


print("Models fitted successfully")
print(f"\nPre-eGFR features: {X_pre.shape[1]}")
print(f"Post-eGFR features: {X_post.shape[1]}")

In [ ]:
from sklearn.metrics import roc_auc_score

# ROC-AUC scores
pre_auc = roc_auc_score(y_pre, model_pre.predict_proba(X_pre)[:,1])
post_auc = roc_auc_score(y_post, model_post.predict_proba(X_post)[:,1])

print(f"Pre-eGFR ROC-AUC: {pre_auc:.3f}")
print(f"Post-eGFR ROC-AUC: {post_auc:.3f}")

# extract odds ratios
odds_ratios_pre = pd.DataFrame({
    'Feature': X_pre.columns,
    'Odds_Ratio': np.exp(model_pre.named_steps['model'].coef_[0]),
    'Coefficient': model_pre.named_steps['model'].coef_[0]
}).sort_values('Odds_Ratio', ascending=False)

odds_ratios_post = pd.DataFrame({
    'Feature': X_post.columns,
    'Odds_Ratio': np.exp(model_post.named_steps['model'].coef_[0]),
    'Coefficient': model_post.named_steps['model'].coef_[0]
}).sort_values('Odds_Ratio', ascending=False)

print("\nPre-eGFR Odds Ratios:")
print(odds_ratios_pre.to_string(index=False))

print("\nPost-eGFR Odds Ratios:")
print(odds_ratios_post.to_string(index=False))

In [ ]:
import joblib

# save models
joblib.dump(model_pre, '../models/stage4_pre_egfr.pkl')
joblib.dump(model_post, '../models/stage4_post_egfr.pkl')

# save odds ratios
odds_ratios_pre.to_csv('../outputs/odds_ratios_pre_egfr.csv', index=False)
odds_ratios_post.to_csv('../outputs/odds_ratios_post_egfr.csv', index=False)



### Sensitivity Analysis 

In [ ]:
df_wl_all =  pd.read_csv('../data/wl_clean.csv')

#put back in active waitlist patients as 0
df_wl_all['RECEIVED_TX'] = df_wl_all['RECEIVED_TX'].fillna(0)

# apply insurance mapping
insurance_map = {
    1.0: 'Private', 2.0: 'Medicare', 3.0: 'Medicaid',
    4.0: 'Medicaid', 5.0: 'Other', 6.0: 'Other',
    7.0: 'Other', 8.0: 'Other', 9.0: 'Other',
    10.0: 'Other', 11.0: 'Other', 12.0: 'Other',
    13.0: 'Medicare', 14.0: 'Medicare',
}
df_wl_all['INSURANCE_GRP'] = df_wl_all['CAN_PRIMARY_PAY'].map(insurance_map)

# rebuild preemptive flag
df_wl_all['PREEMPTIVE'] = (df_wl_all['CAN_ON_DIAL'] == 'N').astype(int)

# rebuild eGFR era flag
df_wl_all['CAN_LISTING_DT'] = pd.to_datetime(df_wl_all['CAN_LISTING_DT'])
df_wl_all['EGFR_ERA_WL'] = (df_wl_all['CAN_LISTING_DT'] >= '2021-09-01').astype(int)

print(f"Full dataset with active patients recoded: {len(df_wl_all):,}")
print(f"Transplant rate with active patients as 0: {df_wl_all['RECEIVED_TX'].mean():.1%}")

In [ ]:
# split into pre/post eGFR cohorts
df_sens_pre = df_wl_all[df_wl_all['EGFR_ERA_WL'] == 0].copy()
df_sens_post = df_wl_all[df_wl_all['EGFR_ERA_WL'] == 1].copy()

# drop missing
df_sens_pre = df_sens_pre[covariates + ['RECEIVED_TX']].dropna()
df_sens_post = df_sens_post[covariates + ['RECEIVED_TX']].dropna()

print(f"Sensitivity pre-eGFR: {len(df_sens_pre):,}")
print(f"Sensitivity post-eGFR: {len(df_sens_post):,}")

In [ ]:
# prepare features
X_sens_pre, y_sens_pre = prepare_features(df_sens_pre)
X_sens_post, y_sens_post = prepare_features(df_sens_post)

# fit sensitivity models
sens_model_pre = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

sens_model_post = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

sens_model_pre.fit(X_sens_pre, y_sens_pre)
sens_model_post.fit(X_sens_post, y_sens_post)

print("\nSensitivity models fitted successfully")

In [ ]:
# extract sensitivity odds ratios
sens_or_pre = pd.DataFrame({
    'Feature': X_sens_pre.columns,
    'Sens_OR_Pre': np.exp(sens_model_pre.named_steps['model'].coef_[0])
})

sens_or_post = pd.DataFrame({
    'Feature': X_sens_post.columns,
    'Sens_OR_Post': np.exp(sens_model_post.named_steps['model'].coef_[0])
})

# merge with primary odds ratios for comparison
comparison_pre = odds_ratios_pre[['Feature', 'Odds_Ratio']].merge(
    sens_or_pre, on='Feature'
).rename(columns={'Odds_Ratio': 'Primary_OR_Pre'})

comparison_post = odds_ratios_post[['Feature', 'Odds_Ratio']].merge(
    sens_or_post, on='Feature'
).rename(columns={'Odds_Ratio': 'Primary_OR_Post'})

# filter to key variables only
key_vars = ['CAN_RACE_SRTR_BLACK', 'CAN_RACE_SRTR_WHITE', 
            'CAN_RACE_SRTR_NATIVE', 'INSURANCE_GRP_Private', 
            'INSURANCE_GRP_Medicare', 'PREEMPTIVE']

print("Pre-eGFR: Primary vs Sensitivity Odds Ratios")
print(comparison_pre[comparison_pre['Feature'].isin(key_vars)].to_string(index=False))

print("\nPost-eGFR: Primary vs Sensitivity Odds Ratios")
print(comparison_post[comparison_post['Feature'].isin(key_vars)].to_string(index=False))

In [ ]:
comparison_pre.to_csv('../outputs/sensitivity_comparison_pre.csv', index=False)
comparison_post.to_csv('../outputs/sensitivity_comparison_post.csv', index=False)
print("Sensitivity results saved")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
# get predicted probabilities
y_pre_prob = model_pre.predict_proba(X_pre)[:,1]
y_post_prob = model_post.predict_proba(X_post)[:,1]

# compute ROC curves
fpr_pre, tpr_pre, _ = roc_curve(y_pre, y_pre_prob)
fpr_post, tpr_post, _ = roc_curve(y_post, y_post_prob)

# plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_pre, tpr_pre, label=f'Pre-eGFR (AUC = {pre_auc:.3f})', color='steelblue')
ax.plot(fpr_post, tpr_post, label=f'Post-eGFR (AUC = {post_auc:.3f})', color='coral')
ax.plot([0,1], [0,1], 'k--', label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Stage 4 Transplant Receipt')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/roc_curve_stage4.png', dpi=300)
plt.show()


In [ ]:
from scipy import stats

def get_or_with_ci(model, X, feature_names, z=1.96):
    coefs = model.named_steps['model'].coef_[0]
    or_vals = np.exp(coefs)
    
    # use standard logistic regression SE approximation
    X_scaled = model.named_steps['scaler'].transform(X)
    p = model.predict_proba(X)[:,1]
    W = p * (1 - p)
    
    # compute SE column by column to avoid large matrix
    se = np.zeros(len(coefs))
    for j in range(X_scaled.shape[1]):
        se[j] = 1 / np.sqrt(np.sum(W * X_scaled[:, j]**2))
    
    ci_lower = np.exp(coefs - z * se)
    ci_upper = np.exp(coefs + z * se)
    
    return pd.DataFrame({
        'Feature': feature_names,
        'OR': or_vals,
        'CI_Lower': ci_lower,
        'CI_Upper': ci_upper
    })

or_pre_ci = get_or_with_ci(model_pre, X_pre, X_pre.columns)
or_post_ci = get_or_with_ci(model_post, X_post, X_post.columns)

print("Pre-eGFR with CIs:")
print(or_pre_ci[or_pre_ci['Feature'].isin(key_vars)].to_string(index=False))

In [ ]:
key_vars = [
    'CAN_RACE_SRTR_BLACK',
    'CAN_RACE_SRTR_NATIVE',
    'CAN_RACE_SRTR_PACIFIC',
    'CAN_RACE_SRTR_WHITE',
    'CAN_RACE_SRTR_MULTI',
    'INSURANCE_GRP_Private',
    'INSURANCE_GRP_Medicare',
    'PREEMPTIVE',
]

label_map = {
    'CAN_RACE_SRTR_BLACK': 'Black',
    'CAN_RACE_SRTR_NATIVE': 'Native American',
    'CAN_RACE_SRTR_PACIFIC': 'Pacific Islander',
    'CAN_RACE_SRTR_WHITE': 'White',
    'CAN_RACE_SRTR_MULTI': 'Multiracial',
    'INSURANCE_GRP_Private': 'Private Insurance',
    'INSURANCE_GRP_Medicare': 'Medicare',
    'PREEMPTIVE': 'Pre-emptive Listing',
}

# filter and label
pre_plot = or_pre_ci[or_pre_ci['Feature'].isin(key_vars)].copy()
post_plot = or_post_ci[or_post_ci['Feature'].isin(key_vars)].copy()
pre_plot['Label'] = pre_plot['Feature'].map(label_map)
post_plot['Label'] = post_plot['Feature'].map(label_map)

# sort by pre-eGFR OR
pre_plot = pre_plot.sort_values('OR')

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, df, title, color in zip(
    axes,
    [pre_plot, post_plot],
    ['Pre-eGFR Era', 'Post-eGFR Era'],
    ['steelblue', 'coral']
):
    y = range(len(df))
    ax.axvline(x=1, color='black', linewidth=1.5, linestyle='--', label='Reference (Asian / Medicaid)')
    
    for i, row in enumerate(df.itertuples()):
        ax.plot([row.CI_Lower, row.CI_Upper], [i, i], color=color, alpha=0.6, linewidth=2)
        ax.scatter(row.OR, i, color=color, zorder=5, s=80)
    
    ax.set_yticks(list(y))
    ax.set_yticklabels(df['Label'])
    ax.set_xlabel('Odds Ratio\n← Less likely to receive transplant | More likely →')
    ax.set_title(title)
    ax.axvspan(0, 1, alpha=0.03, color='red')
    ax.axvspan(1, ax.get_xlim()[1] if ax.get_xlim()[1] > 1 else 1.2, alpha=0.03, color='green')
    ax.set_xlim(0.85, 1.15)

axes[0].set_ylabel('Variable (Reference: Asian race, Medicaid insurance)')
fig.suptitle('Adjusted Odds Ratios — Stage 4 Transplant Receipt', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/forest_plot_stage4.png', dpi=300)
plt.show()


In [ ]:
#FIXED GRAPHIC 
# force same order on both plots
pre_plot = pre_plot.sort_values('OR')
post_plot['Label'] = post_plot['Feature'].map(label_map)
post_plot = post_plot.set_index('Label').loc[pre_plot['Label'].values].reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, df, title, color in zip(
    axes,
    [pre_plot, post_plot],
    ['Pre-eGFR Era', 'Post-eGFR Era'],
    ['steelblue', 'coral']
):
    y = range(len(df))
    ax.axvline(x=1, color='black', linewidth=1.5, linestyle='--')
    
    for i, row in enumerate(df.itertuples()):
        ax.plot([row.CI_Lower, row.CI_Upper], [i, i], color=color, alpha=0.6, linewidth=2)
        ax.scatter(row.OR, i, color=color, zorder=5, s=80)
    
    ax.set_yticks(list(y))
    ax.set_yticklabels(df['Label'])
    ax.set_xlabel('Odds Ratio\n← Less likely to receive transplant | More likely →')
    ax.set_title(title)
    ax.axvspan(0.85, 1, alpha=0.03, color='red')
    ax.axvspan(1, 1.15, alpha=0.03, color='green')
    ax.set_xlim(0.85, 1.15)

axes[0].set_ylabel('Variable (Reference: Asian race, Medicaid insurance)')
fig.suptitle('Adjusted Odds Ratios — Stage 4 Transplant Receipt', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/forest_plot_stage4.png', dpi=300)
plt.show()


In [ ]:
# check reference groups
print("Race categories (alphabetical order):")
print(sorted(df_pre['CAN_RACE_SRTR'].unique()))

print("\nInsurance categories (alphabetical order):")
print(sorted(df_pre['INSURANCE_GRP'].unique()))

print("\nDummy columns in model — these are NON-reference groups:")
print([col for col in X_pre.columns if 'RACE' in col or 'INSURANCE' in col])